# CNN-Att-LSTM: Uncertainty-Aware IoT Intrusion Detection
### ⚠️ BEFORE RUNNING ANYTHING:
### Go to **Runtime → Change runtime type → T4 GPU → Save**

---
## CELL 1 — Check GPU
Run this first. You should see a GPU name, not 'CPU only'.

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU ready: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > T4 GPU')

---
## CELL 2 — Install packages
This takes about 1-2 minutes. Wait until you see 'Done'.

In [ ]:
!pip install -q imbalanced-learn kaggle
import importlib, subprocess
for pkg in ['sklearn','pandas','numpy','matplotlib','seaborn','torch']:
    try:
        importlib.import_module(pkg)
        print(f'  {pkg}: OK')
    except ImportError:
        print(f'  {pkg}: installing...')
        subprocess.run(['pip','install','-q',pkg])
print('Done!')

---
## CELL 3 — Setup Kaggle API

### How to find your username and API key:
1. Go to https://kaggle.com and log in
2. Click your **profile picture** (top right) → **Settings**
3. Your **username** is shown at the top of the Settings page
4. Scroll down to the **API** section → click **Create New Token**
5. A file called `kaggle.json` downloads — open it with Notepad
6. You will see: `{"username":"abc","key":"xyz123..."}` — copy those two values
7. Paste them into the two lines in the next cell

In [ ]:
import os, json

# ✏️ FILL IN YOUR DETAILS HERE (replace the placeholder text)
KAGGLE_USERNAME = "your_kaggle_username"   # e.g. "joyabiswas"
KAGGLE_KEY      = "your_kaggle_api_key"    # long string of letters and numbers

# ---- Do not edit below this line ----
os.makedirs('/root/.kaggle', exist_ok=True)
creds = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Verify it works
result = os.popen('kaggle datasets list --max-size 1 2>&1').read()
if 'Unauthorized' in result or '401' in result or 'Could not find' in result:
    print('ERROR: Username or API key is wrong. Double-check and try again.')
    print(result)
else:
    print('Kaggle API configured successfully! Ready to download datasets.')

---
## CELL 4 — Download Datasets
This downloads CIC-IoT-2023 and N-BaIoT automatically.
Takes 3-8 minutes depending on Colab speed.

In [ ]:
import os

os.makedirs('/content/datasets', exist_ok=True)

print('Downloading CIC-IoT-2023 (combined CSV version)...')
ret1 = os.system('kaggle datasets download -d akashdogra/ciciot23csv -p /content/datasets/ciciot --unzip --quiet')
if ret1 != 0:
    print('  Trying alternative source...')
    os.system('kaggle datasets download -d akashdogra/cic-iot-2023 -p /content/datasets/ciciot --unzip --quiet')

print('Downloading N-BaIoT...')
ret2 = os.system('kaggle datasets download -d mkashifn/nbaiot-dataset -p /content/datasets/nbaiot --unzip --quiet')

print('\nDownload complete! Files found:')
for root, dirs, fnames in os.walk('/content/datasets'):
    for f in fnames:
        if f.endswith('.csv'):
            full = os.path.join(root, f)
            size = os.path.getsize(full) // 1024
            print(f'  {full} ({size} KB)')

---
## CELL 5 — Load CIC-IoT-2023

In [ ]:
import pandas as pd
import numpy as np
import glob, os
import warnings
warnings.filterwarnings('ignore')

csv_files = glob.glob('/content/datasets/ciciot/**/*.csv', recursive=True)
print(f'Found {len(csv_files)} CSV files in CIC-IoT-2023')

dfs = []
for f in csv_files:
    try:
        tmp = pd.read_csv(f, low_memory=False, nrows=50000)
        dfs.append(tmp)
        print(f'  Loaded: {os.path.basename(f)}  shape={tmp.shape}')
    except Exception as e:
        print(f'  Skipped {os.path.basename(f)}: {e}')

ciciot = pd.concat(dfs, ignore_index=True)
print(f'\nTotal CIC-IoT-2023 shape: {ciciot.shape}')
print(ciciot.dtypes.value_counts())

---
## CELL 6 — Find Label Column and Show Class Distribution

In [ ]:
# Auto-detect label column
def find_label_col(df):
    keywords = ['label', 'attack', 'class', 'target', 'type']
    for kw in keywords:
        for col in df.columns:
            if kw in col.lower():
                return col
    # Fallback: last column if it has few unique values
    last = df.columns[-1]
    if df[last].nunique() < 50:
        return last
    return None

CICIOT_LABEL = find_label_col(ciciot)
print(f'Label column detected: "{CICIOT_LABEL}"')
print(f'\nClass distribution:')
print(ciciot[CICIOT_LABEL].value_counts())

---
## CELL 7 — Preprocess CIC-IoT-2023

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

def preprocess_df(df, label_col, max_samples=150000):
    df = df.copy()

    # Sample if too large
    if len(df) > max_samples:
        df = df.sample(n=max_samples, random_state=42).reset_index(drop=True)

    # Separate label
    y_raw = df[label_col].astype(str)
    df    = df.drop(columns=[label_col])

    # Drop non-numeric columns
    df = df.select_dtypes(include=[np.number])

    # Clean infinities and NaN
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(df.median())

    # Encode labels
    le = LabelEncoder()
    y  = le.fit_transform(y_raw)

    # Scale features
    scaler = StandardScaler()
    X = scaler.fit_transform(df.values)

    print(f'X: {X.shape}  |  Classes ({len(le.classes_)}): {list(le.classes_)}')
    return X, y, le, scaler, list(df.columns)

X, y, le, scaler, feature_names = preprocess_df(ciciot, CICIOT_LABEL)
NUM_CLASSES = len(le.classes_)
INPUT_SIZE  = X.shape[1]
print(f'Input size: {INPUT_SIZE}, Num classes: {NUM_CLASSES}')

---
## CELL 8 — Apply ADASYN (Handle Class Imbalance)

In [ ]:
from imblearn.over_sampling import ADASYN
from collections import Counter

print('Class distribution BEFORE ADASYN:')
for cls, cnt in sorted(Counter(y).items()):
    print(f'  {le.classes_[cls]}: {cnt}')

try:
    adasyn = ADASYN(random_state=42, n_neighbors=5)
    X_bal, y_bal = adasyn.fit_resample(X, y)
    print(f'\nClass distribution AFTER ADASYN:')
    for cls, cnt in sorted(Counter(y_bal).items()):
        print(f'  {le.classes_[cls]}: {cnt}')
except Exception as e:
    print(f'ADASYN note: {e}')
    print('Using original (already balanced enough)')
    X_bal, y_bal = X, y

print(f'\nFinal dataset: {X_bal.shape}')

---
## CELL 9 — Train/Test Split and DataLoaders

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

# Remove classes with fewer than 2 samples (stratify requires at least 2)
from collections import Counter
counts = Counter(y_bal)
valid_classes = {cls for cls, cnt in counts.items() if cnt >= 2}
mask = np.array([yi in valid_classes for yi in y_bal])
X_bal_f = X_bal[mask]
y_bal_f = y_bal[mask]
removed = len(y_bal) - len(y_bal_f)
if removed > 0:
    print(f'Removed {removed} samples from classes with <2 members')

X_tr, X_te, y_tr, y_te = train_test_split(
    X_bal_f, y_bal_f, test_size=0.2, random_state=42, stratify=y_bal_f
)

def make_loader(X, y, batch=512, shuffle=True):
    Xt = torch.FloatTensor(X).to(device)
    yt = torch.LongTensor(y).to(device)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch, shuffle=shuffle)

train_loader = make_loader(X_tr, y_tr, shuffle=True)
test_loader  = make_loader(X_te, y_te, shuffle=False)

# Update num classes in case any were dropped
NUM_CLASSES = len(np.unique(y_bal_f))
print(f'Train: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'Num classes: {NUM_CLASSES}')

---
## CELL 10 — Define CNN-Att-LSTM Architecture

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SelfAttention(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.W = nn.Linear(hidden, hidden)
        self.v = nn.Linear(hidden, 1, bias=False)

    def forward(self, x):
        scores  = self.v(torch.tanh(self.W(x))).squeeze(-1)
        weights = F.softmax(scores, dim=-1)
        context = (x * weights.unsqueeze(-1)).sum(dim=1)
        return context, weights


class CNN_Att_LSTM(nn.Module):
    def __init__(self, input_size, num_classes, dropout=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.bn    = nn.BatchNorm1d(128)
        self.pool  = nn.MaxPool1d(2)
        self.attn  = SelfAttention(128)
        self.lstm  = nn.LSTM(128, 256, num_layers=2,
                             batch_first=True, dropout=dropout)
        self.drop  = nn.Dropout(p=dropout)
        self.fc1   = nn.Linear(256, 128)
        self.fc2   = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.bn(x)
        x = x.permute(0, 2, 1)
        _, weights = self.attn(x)
        x = x * weights.unsqueeze(-1)
        _, (hn, _) = self.lstm(x)
        x = hn[-1]
        x = self.drop(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


# NUM_CLASSES is updated from Cell 9
model = CNN_Att_LSTM(INPUT_SIZE, NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model ready. Input size: {INPUT_SIZE}, Classes: {NUM_CLASSES}')
print(f'Total parameters: {total_params:,}')

---
## CELL 11 — Train (takes ~10-20 minutes with GPU)

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

EPOCHS = 30
history = {'loss': [], 'acc': []}
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    total_loss = 0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    # --- Evaluate ---
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for Xb, yb in test_loader:
            preds = model(Xb).argmax(1)
            correct += (preds == yb).sum().item()
            total   += yb.size(0)

    avg_loss = total_loss / len(train_loader)
    acc      = 100 * correct / total
    scheduler.step(avg_loss)
    history['loss'].append(avg_loss)
    history['acc'].append(acc)

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), '/content/best_cnn_att_lstm.pth')

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{EPOCHS}  Loss={avg_loss:.4f}  Acc={acc:.2f}%  Best={best_acc:.2f}%')

print(f'\nTraining complete! Best accuracy: {best_acc:.2f}%')

---
## CELL 12 — Full Evaluation (Accuracy, F1, Precision, Recall)

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score,
                              precision_score, recall_score)

model.load_state_dict(torch.load('/content/best_cnn_att_lstm.pth'))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        preds = model(Xb).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

acc = accuracy_score(all_true, all_preds)
f1  = f1_score(all_true, all_preds, average='weighted', zero_division=0)
pre = precision_score(all_true, all_preds, average='weighted', zero_division=0)
rec = recall_score(all_true, all_preds, average='weighted', zero_division=0)

print('=' * 50)
print('       RESULTS — CIC-IoT-2023')
print('=' * 50)
print(f'Accuracy:   {acc*100:.2f}%')
print(f'Precision:  {pre*100:.2f}%')
print(f'Recall:     {rec*100:.2f}%')
print(f'F1 Score:   {f1*100:.2f}%')
print('=' * 50)
print()
print(classification_report(all_true, all_preds,
                             target_names=le.classes_, zero_division=0))

---
## CELL 13 — Confusion Matrix Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix — CNN-Att-LSTM (CIC-IoT-2023)', pad=15)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()
print('Saved to /content/confusion_matrix.png')

---
## CELL 14 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'], 'b-o', markersize=4, linewidth=1.5)
ax1.set_title('Training Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(history['acc'], 'g-o', markersize=4, linewidth=1.5)
ax2.set_title('Validation Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print('Saved to /content/training_curves.png')

---
## CELL 15 — Uncertainty Quantification (Monte Carlo Dropout)
This is the KEY novel contribution of this paper.

In [ ]:
def mc_predict(model, X_np, n_passes=50, batch_size=256):
    """
    Monte Carlo Dropout inference.
    Keeps dropout ACTIVE during prediction to estimate uncertainty.
    Returns: mean_probs (N, C), std_probs (N, C)
    """
    model.train()  # dropout stays ON
    X_t  = torch.FloatTensor(X_np).to(device)
    runs = []
    with torch.no_grad():
        for _ in range(n_passes):
            probs_pass = []
            for i in range(0, len(X_t), batch_size):
                logits = model(X_t[i:i+batch_size])
                probs_pass.append(F.softmax(logits, dim=1).cpu().numpy())
            runs.append(np.vstack(probs_pass))
    runs = np.stack(runs)           # (passes, N, C)
    return runs.mean(axis=0), runs.std(axis=0)

# Use 2000 test samples
sample_n = min(2000, len(X_te))
idx      = np.random.choice(len(X_te), sample_n, replace=False)

print(f'Running 50 Monte Carlo passes on {sample_n} samples...')
mean_p, std_p = mc_predict(model, X_te[idx], n_passes=50)

mc_pred  = mean_p.argmax(axis=1)
mc_conf  = mean_p.max(axis=1)        # confidence per sample
mc_unc   = std_p.max(axis=1)         # uncertainty per sample
y_sample = y_te[idx]

mc_acc = accuracy_score(y_sample, mc_pred)
print(f'\nMC Dropout Results:')
print(f'  Accuracy:          {mc_acc*100:.2f}%')
print(f'  Mean Confidence:   {mc_conf.mean()*100:.1f}%')
print(f'  Mean Uncertainty:  {mc_unc.mean():.4f}')

# High vs low confidence split
hi = mc_conf >= 0.90
lo = mc_conf <  0.90
if hi.sum() > 0:
    print(f'  High-conf (>=90%): {hi.sum()} samples → Acc {accuracy_score(y_sample[hi], mc_pred[hi])*100:.2f}%')
if lo.sum() > 0:
    print(f'  Low-conf  (<90%):  {lo.sum()} samples → Acc {accuracy_score(y_sample[lo], mc_pred[lo])*100:.2f}%')

---
## CELL 16 — Uncertainty Visualization

In [ ]:
correct = (mc_pred == y_sample)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Plot 1: Confidence histogram
axes[0].hist(mc_conf[correct],  bins=40, alpha=0.7, color='green', label='Correct')
axes[0].hist(mc_conf[~correct], bins=40, alpha=0.7, color='red',   label='Wrong')
axes[0].axvline(0.90, color='black', linestyle='--', label='90% threshold')
axes[0].set_title('Confidence Distribution\n(MC Dropout, 50 passes)')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].legend()

# Plot 2: Uncertainty vs confidence
axes[1].scatter(mc_unc[correct],  mc_conf[correct],
                alpha=0.3, c='green', s=8, label='Correct')
axes[1].scatter(mc_unc[~correct], mc_conf[~correct],
                alpha=0.4, c='red',   s=8, label='Wrong')
axes[1].set_title('Uncertainty vs Confidence\nper Prediction')
axes[1].set_xlabel('Uncertainty (Std Dev across passes)')
axes[1].set_ylabel('Mean Confidence')
axes[1].legend()

plt.tight_layout()
plt.savefig('/content/uncertainty_plot.png', dpi=150)
plt.show()
print('Saved to /content/uncertainty_plot.png')

---
## CELL 17 — Cross-Dataset Evaluation on N-BaIoT

In [ ]:
nb_files = glob.glob('/content/datasets/nbaiot/**/*.csv', recursive=True)
print(f'N-BaIoT files found: {len(nb_files)}')
for f in nb_files[:5]:
    print(f'  {f}')

In [ ]:
if nb_files:
    # Load N-BaIoT samples
    nb_dfs = []
    for f in nb_files[:20]:
        try:
            tmp = pd.read_csv(f, low_memory=False, nrows=5000)
            # N-BaIoT has no label column — derive from filename
            fname = os.path.basename(os.path.dirname(f))
            tmp['label'] = 'benign' if 'benign' in f.lower() else fname
            nb_dfs.append(tmp)
        except:
            pass

    nbaiot = pd.concat(nb_dfs, ignore_index=True)
    print(f'N-BaIoT shape: {nbaiot.shape}')
    print(nbaiot['label'].value_counts())
else:
    print('N-BaIoT not loaded. Skipping cross-dataset section.')
    nbaiot = None

In [ ]:
if nbaiot is not None:
    nb_clean = nbaiot.select_dtypes(include=[np.number]).copy()
    nb_clean = nb_clean.replace([np.inf, -np.inf], np.nan).fillna(0)

    y_nb_bin = (nbaiot['label'] != 'benign').astype(int).values

    # Align feature size to what model expects
    nb_arr = nb_clean.values
    if nb_arr.shape[1] < INPUT_SIZE:
        nb_arr = np.hstack([nb_arr, np.zeros((nb_arr.shape[0], INPUT_SIZE - nb_arr.shape[1]))])
    else:
        nb_arr = nb_arr[:, :INPUT_SIZE]

    nb_arr = StandardScaler().fit_transform(nb_arr)

    # Sample
    n = min(20000, len(nb_arr))
    idx = np.random.choice(len(nb_arr), n, replace=False)
    nb_arr   = nb_arr[idx]
    y_nb_bin = y_nb_bin[idx]

    # Predict with model (no retraining)
    model.eval()
    nb_t  = torch.FloatTensor(nb_arr).to(device)
    nb_dl = DataLoader(TensorDataset(nb_t), batch_size=512)
    preds_nb = []
    with torch.no_grad():
        for (Xb,) in nb_dl:
            preds_nb.extend(model(Xb).argmax(1).cpu().numpy())

    # Binary: 0=benign, 1=attack
    preds_nb_bin = (np.array(preds_nb) > 0).astype(int)

    cross_acc = accuracy_score(y_nb_bin, preds_nb_bin)
    cross_f1  = f1_score(y_nb_bin, preds_nb_bin, average='weighted', zero_division=0)
    cross_pre = precision_score(y_nb_bin, preds_nb_bin, average='weighted', zero_division=0)
    cross_rec = recall_score(y_nb_bin, preds_nb_bin, average='weighted', zero_division=0)

    print('=' * 50)
    print('  CROSS-DATASET RESULTS — N-BaIoT')
    print('  (Model trained on CIC-IoT-2023 only)')
    print('=' * 50)
    print(f'  Accuracy:   {cross_acc*100:.2f}%')
    print(f'  Precision:  {cross_pre*100:.2f}%')
    print(f'  Recall:     {cross_rec*100:.2f}%')
    print(f'  F1 Score:   {cross_f1*100:.2f}%')
    print('=' * 50)

---
## CELL 18 — FINAL SUMMARY (Screenshot this for your paper)

In [ ]:
print('\n' + '=' * 55)
print('       PAPER RESULTS SUMMARY')
print('       CNN-Att-LSTM with MC Dropout + ADASYN')
print('=' * 55)
print(f'  Dataset 1: CIC-IoT-2023')
print(f'    Accuracy:   {acc*100:.2f}%')
print(f'    Precision:  {pre*100:.2f}%')
print(f'    Recall:     {rec*100:.2f}%')
print(f'    F1 Score:   {f1*100:.2f}%')
print()
print(f'  Uncertainty Analysis (MC Dropout):')
print(f'    MC Accuracy:        {mc_acc*100:.2f}%')
print(f'    Mean Confidence:    {mc_conf.mean()*100:.1f}%')
print(f'    Mean Uncertainty:   {mc_unc.mean():.4f}')
if nbaiot is not None:
    print()
    print(f'  Dataset 2: N-BaIoT (cross-dataset, no retraining)')
    print(f'    Accuracy:   {cross_acc*100:.2f}%')
    print(f'    F1 Score:   {cross_f1*100:.2f}%')
print()
print('  Saved figures:')
print('    /content/confusion_matrix.png')
print('    /content/training_curves.png')
print('    /content/uncertainty_plot.png')
print('=' * 55)
print()
print('  SCREENSHOT THESE RESULTS and share with Claude.')
print('  The full 10-page paper will be written for you!')

---
## CELL 19 — Download all figures to your computer

In [ ]:
from google.colab import files
import os

for fig_file in ['confusion_matrix.png', 'training_curves.png', 'uncertainty_plot.png']:
    path = f'/content/{fig_file}'
    if os.path.exists(path):
        files.download(path)
        print(f'Downloading: {fig_file}')
    else:
        print(f'Not found: {fig_file} (run previous cells first)')